In [1]:
pip install langchain langchain-openai pypdf 


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 41.3 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.23.0
    Uninstalling openai-2.23.0:
      Successfully uninstalled openai-2.23.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
Note: you may need to restart the kernel to use updated packages.


In [8]:
!pip install -q langchain langchain-community langchain-text-splitters google-generativeai

In [3]:
!pip install --upgrade google-generativeai

In [9]:
!pip install -q langchain-text-splitters

In [13]:
import google.generativeai as genai
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
gemini_key = user_secrets.get_secret("GEMINI_API_KEY")

genai.configure(api_key=gemini_key)

# Updated model name (working in 2026)
model = genai.GenerativeModel('gemini-2.5-flash')   # or 'gemini-2.5-flash-l
print("Gemini model initialized with gemini-2.5-flash")

Gemini model initialized with gemini-2.5-flash


In [5]:
import os

# Create directory
os.makedirs('company_docs', exist_ok=True)

# HR Policy
with open('company_docs/hr_policy.txt', 'w') as f:
    f.write("""Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. 
Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 
6 weeks paid leave for secondary caregivers.""")

# Benefits Policy
with open('company_docs/benefits.txt', 'w') as f:
    f.write("""Benefits Information
Health Insurance: Company covers 100% of health insurance premium for employees.
401(k) Matching: Up to 5% matching contribution.
Sick Leave: Unlimited sick leave with manager approval.""")

# IT Policy
with open('company_docs/it_policy.txt', 'w') as f:
    f.write("""IT Department Policy
Laptop Policy: Employees get a company laptop. Must be returned upon leaving.
Software: Only approved software can be installed.
Security: Multi-factor authentication is mandatory.""")

print("✅ Sample documents created!")
print(os.listdir('company_docs'))

✅ Sample documents created!
['it_policy.txt', 'benefits.txt', 'hr_policy.txt']


In [6]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    'company_docs/',
    glob='*.txt',
    loader_cls=TextLoader
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")
for doc in documents:
    print(f"File: {doc.metadata['source']}")
    print(f"Preview: {doc.page_content[:150]}...\n")

/tmp/ipykernel_57/2576550220.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 3 documents
File: company_docs/it_policy.txt
Preview: IT Department Policy
Laptop Policy: Employees get a company laptop. Must be returned upon leaving.
Software: Only approved software can be installed.
...

File: company_docs/benefits.txt
Preview: Benefits Information
Health Insurance: Company covers 100% of health insurance premium for employees.
401(k) Matching: Up to 5% matching contribution....

File: company_docs/hr_policy.txt
Preview: Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and ...



In [10]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=['\n\n', '\n', '. ', ' ', '']
)

chunks = text_splitter.split_documents(documents)

print(f"Split into {len(chunks)} chunks")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1}:")
    print(chunk.page_content)
    print(f"Length: {len(chunk.page_content)} chars")

Split into 3 chunks

Chunk 1:
IT Department Policy
Laptop Policy: Employees get a company laptop. Must be returned upon leaving.
Software: Only approved software can be installed.
Security: Multi-factor authentication is mandatory.
Length: 201 chars

Chunk 2:
Benefits Information
Health Insurance: Company covers 100% of health insurance premium for employees.
401(k) Matching: Up to 5% matching contribution.
Sick Leave: Unlimited sick leave with manager approval.
Length: 206 chars

Chunk 3:
Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. 
Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 
6 weeks paid leave for secondary caregivers.
Length: 404 chars


In [11]:
def simple_search(query, chunks, top_k=3):
    query_lower = query.lower()
    scored_chunks = []
    
    for chunk in chunks:
        content_lower = chunk.page_content.lower()
        score = 0
        for word in query_lower.split():
            score += content_lower.count(word)
        if score > 0:
            scored_chunks.append((score, chunk))
    
    scored_chunks.sort(reverse=True, key=lambda x: x[0])
    return [chunk for score, chunk in scored_chunks[:top_k]]

# Test
query = "What is the vacation policy?"
relevant = simple_search(query, chunks)
print(f"Found {len(relevant)} relevant chunks for: {query}")
for i, chunk in enumerate(relevant):
    print(f"\n--- Chunk {i+1} ---")
    print(chunk.page_content)

Found 2 relevant chunks for: What is the vacation policy?

--- Chunk 1 ---
Employee Handbook - HR Policies
Vacation Policy: All full-time employees receive 15 days of paid vacation per year. 
Vacation days accrue monthly and can be used after 90 days.

Remote Work Policy: Employees may work remotely up to 3 days per week. 
Remote work requires manager approval.

Parental Leave: 12 weeks paid parental leave for primary caregivers. 
6 weeks paid leave for secondary caregivers.

--- Chunk 2 ---
IT Department Policy
Laptop Policy: Employees get a company laptop. Must be returned upon leaving.
Software: Only approved software can be installed.
Security: Multi-factor authentication is mandatory.


In [14]:
def rag_query(query, chunks, top_k=3):
    # Retrieve
    relevant_chunks = simple_search(query, chunks, top_k)
    
    if not relevant_chunks:
        return "No relevant information found in the documents."
    
    # Build context
    context = '\n\n---\n\n'.join([chunk.page_content for chunk in relevant_chunks])
    
    # Create prompt
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context provided below.
If the answer is not in the context, say "I don't have information about that in the company documents."

Context:
{context}

Question: {query}

Answer:"""
    
    # Generate answer with Gemini
    response = model.generate_content(prompt)
    return response.text

# Test RAG
questions = [
    "How many vacation days do full-time employees get?",
    "Can employees work remotely?",
    "What is the parental leave policy?",
    "What is the dress code?"   # Should say not in context
]

for q in questions:
    print(f"\n{'='*60}")
    print(f"Q: {q}")
    print(f"{'='*60}")
    answer = rag_query(q, chunks)
    print(f"A: {answer}")


Q: How many vacation days do full-time employees get?
A: All full-time employees receive 15 days of paid vacation per year.

Q: Can employees work remotely?
A: Yes, employees may work remotely up to 3 days per week, and remote work requires manager approval.

Q: What is the parental leave policy?
A: The parental leave policy is 12 weeks paid leave for primary caregivers and 6 weeks paid leave for secondary caregivers.

Q: What is the dress code?
A: I don't have information about that in the company documents.


In [16]:
print("--- BONUS: COMPARE WITH VS WITHOUT RAG ---")
def ask_without_rag(question):
    if 'vacation' in question.lower():
        return "Standard company vacation policies usually offer 10 to 14 days of paid time off (PTO) per year, depending on the country, industry, and employee's tenure."
    return "I am a generic HR assistant. Please check your specific company handbook for details."

test_q = 'How many vacation days do employees get?'
print('WITHOUT RAG (Generic AI Answer):')
print(ask_without_rag(test_q))
print('\nWITH RAG (Your Specific Company Answer):')
print(rag_query(test_q, chunks))


--- BONUS: COMPARE WITH VS WITHOUT RAG ---
WITHOUT RAG (Generic AI Answer):
Standard company vacation policies usually offer 10 to 14 days of paid time off (PTO) per year, depending on the country, industry, and employee's tenure.

WITH RAG (Your Specific Company Answer):
All full-time employees receive 15 days of paid vacation per year.
